# Simplified Transform Analysis
## Least squares nethod for simplified transform application

Simplified transform for bands $A$ and $B$ $(T_{ab}, T_{a}, T_{b})$
 should be applied to instrumental magnitudes using
the following process, where $a_t$  and $b_t$ are instrumnetal 
magnitudes of the target star, $a_c$ and $b_c$ - instrumnetal 
magnitudes of the comparison star, $A_c$ and $B_c$ - standard (catalog)
magnitudes of the comparison star

First, transformed targe color index is calculated:
$$
C_t = A_t - B_t = (A_c - B_c) + T_{ab} ((a_t-b_t) - (a_c-b_c))
$$

This allows for calculating transformed target magnitues:

$$
A_{t} = a_t + (A_c-a_c) + T_a C_t \\
B_{t} = A_t - C_t
$$

alternatively,

$$
B_{t} = b_t + (B_c-b_c) + T_b C_t \\
A_{t} = B_{t} + C_t
$$

It can be combined int overconstrained linear system:

$$\begin{align*}
A_t        &= a_t + (A_c-a_c) + T_a C_t \\
-A_t + B_t &= -C_t \\
B_t        &= b_t + (B_c-b_c) + T_b C_t \\
A_t - B_t  &=  C_t
\end{align*}$$

or, in matrix form

$$
\begin{bmatrix} 1 & 0 \\ -1 & 1 \\ 0 & 1 \\ 1 & -1 \end{bmatrix}
\begin{bmatrix} A_t & B_t  \end{bmatrix} = 
\begin{bmatrix} a_t + (A_c-a_c) + T_a C_t \\ -C_t \\ b_t + (B_c-b_c) + T_b C_t \\ C_t  \end{bmatrix}
$$

Let's provide a function for this algorithm:

In [ ]:
from vsopy.util import MagErr
from vsopy.phot import SimpleTransform
import numpy as np

def apply_simple_transform_lsq(xfm:SimpleTransform,
                           _A_c:MagErr, _B_c:MagErr,
                           _a_c:MagErr, _b_c:MagErr,
                           _a_t:MagErr, _b_t:MagErr) -> tuple[MagErr, MagErr, MagErr, MagErr]:
        Tab, Tab_err = xfm.Tab
        Ta, Ta_err = xfm.Ta
        Tb, Tb_err = xfm.Tb
        Ac, Ac_err = _A_c
        Bc, Bc_err = _B_c
        ac, ac_err = _a_c
        bc, bc_err = _b_c
        at, at_err = _a_t
        bt, bt_err = _b_t

        Ct = Ac - Bc + Tab * ((at-bt) - (ac-bc))

        mA = np.array([
                [1, 0],
                [-1, 1],
                [0, 1],
                [1, -1]
        ])

        vb = np.array([
                at + (Ac-ac) + Ta * Ct,
                -Ct,
                bt + (Bc-bc) + Tb * Ct,
                Ct
        ])

        return np.linalg.lstsq(mA, vb, rcond=None)


In [ ]:
from vsopy.util import ValErr
from measurement import SimpleMeasurement
from experiment import Experiment
from sequence import StarSequence
import numpy as np
from astropy.table import QTable

SEED = 42


EXPERIMENT = Experiment(SEED)


In [ ]:

SEQUENCE = EXPERIMENT.generate_sequence(
    Experiment.GenParams(
        num_stars=20,
        mag=(10.0, 15.0),
        ci=(-0.5, 1.0),
        uncert=ValErr(0.05, 0.05)))
SEQUENCE.stars

In [ ]:
TARGET = EXPERIMENT.create_target([10, 11], [.01, .01], SEQUENCE.bands)
TARGET.stars

In [ ]:
from measurement import SkewedMeasurement
from vsopy.phot import create_simple_transform, apply_simple_transform
from vsopy.util import MagErrDtype

TARGET_MAG = np.linspace(8.0, 17.0, 10)
TARGET_CI = np.linspace(-0.8, 1.8, 9)
MEASUREMENT = EXPERIMENT.create_measurement(
                SkewedMeasurement,
                params = SimpleMeasurement.Params(
                    offset=1.0,
                    scatter=0.05,
                    uncert=ValErr(0.03, 0.03)),
                skew=(-0.1, 0.1))

measured_target = MEASUREMENT(TARGET)
measured_ensemble, measured_comp = EXPERIMENT.measure_ensemble(SEQUENCE, MEASUREMENT)
bands = SEQUENCE.bands
std_A = SEQUENCE.ensemble[SEQUENCE.ensemble['band'] == bands[0]]['M']['mag']
std_B = SEQUENCE.ensemble[SEQUENCE.ensemble['band'] == bands[1]]['M']['mag']
std_comp = SEQUENCE.comp_star
xfm = create_simple_transform(std_A, std_B,
                        measured_ensemble[measured_ensemble['band']
                                          == bands[0]]['instr M']['mag'],
                        measured_ensemble[measured_ensemble['band'] == bands[1]]['instr M']['mag'])

x, res, rank, sv = apply_simple_transform_lsq(xfm,
        std_comp[std_comp['band'] == bands[0]]['M'][0],
        std_comp[std_comp['band'] == bands[1]]['M'][0],
        measured_comp[measured_comp['band'] == bands[0]]['instr M'][0],
        measured_comp[measured_comp['band'] == bands[1]]['instr M'][0],
        measured_target[measured_target['band'] == bands[0]]['instr M'][0],
        measured_target[measured_target['band'] == bands[1]]['instr M'][0],
                           )

simple = apply_simple_transform(xfm,
        std_comp[std_comp['band'] == bands[0]]['M'][0],
        std_comp[std_comp['band'] == bands[1]]['M'][0],
        measured_comp[measured_comp['band'] == bands[0]]['instr M'][0],
        measured_comp[measured_comp['band'] == bands[1]]['instr M'][0],
        measured_target[measured_target['band'] == bands[0]]['instr M'][0],
        measured_target[measured_target['band'] == bands[1]]['instr M'][0],
                           )

# def transform_target(target, seq, num_measurements):
#     measured_target = [MEASUREMENT(target) for _ in range(num_measurements)]
#     measured_ensemble = [EXPERIMENT.measure_ensemble(seq, MEASUREMENT) for _ in range(num_measurements)]
#     bands = SEQUENCE.bands
#     std_A = SEQUENCE.ensemble[seq.ensemble['band'] == bands[0]]['M']['mag']
#     std_B = SEQUENCE.ensemble[seq.ensemble['band'] == bands[1]]['M']['mag']
#     std_comp = SEQUENCE.comp_star
#     result = [apply_simple_transform(
#         create_simple_transform(std_A, std_B,
#                                 ensemble[ensemble['band'] == bands[0]]['instr M']['mag'],
#                                 ensemble[ensemble['band'] == bands[1]]['instr M']['mag']),
#         std_comp[std_comp['band'] == bands[0]]['M'][0],
#         std_comp[std_comp['band'] == bands[1]]['M'][0],
#         comp[comp['band'] == bands[0]]['instr M'][0],
#         comp[comp['band'] == bands[1]]['instr M'][0],
#         target[target['band'] == bands[0]]['instr M'][0],
#         target[target['band'] == bands[1]]['instr M'][0],
#     ) for (ensemble, comp), target in zip(measured_ensemble, measured_target)]
#     return QTable([{'A': row[0], 'B': row[3], 'A1': row[2], 'B1': row[1]} for row in result],
#                   dtype=[MagErrDtype] * 4)

# transform_target(TARGET, SEQUENCE, 3)
s = np.array([simple[0].mag, simple[1].mag])
t =  TARGET.stars['M']['mag']
np.sqrt(np.square(x-t)), np.sqrt(np.square(s-t))

In [ ]:
def target_stats(tv, std):
    return {
        'A': np.mean(tv['A']['mag'] - std[0]),
        'A.std': np.std(tv['A']['mag']),
        'A.err': np.mean(tv['A']['err']),
        'B': np.mean(tv['B']['mag'] - std[1]),
        'B.std': np.std(tv['B']['mag']),
        'B.err': np.mean(tv['B']['err'])
    }

TG_STATS = QTable([target_stats(
    transform_target(
        EXPERIMENT.create_target(
            (mag_A, mag_A+ci),
            (.01, .01),
            SEQUENCE.bands),
        SEQUENCE,
        100),
    (mag_A, mag_A+ci)) | {'std A': mag_A, 'std CI': ci}
 for mag_A in TARGET_MAG for ci in TARGET_CI])

TG_STATS

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

X = np.array(TG_STATS['std A']).reshape(len(TARGET_MAG), len(TARGET_CI))
Y = np.array(TG_STATS['std CI']).reshape(len(TARGET_MAG), len(TARGET_CI))

PREFIX = ['A', 'B']
SUFFIX = ['', '.std', '.err']
FIG, AXS = plt.subplots(len(PREFIX), len(SUFFIX), figsize=(10.24, 10.24), constrained_layout=True)


def plot_contour(ax, column):
    z = np.array(TG_STATS[column]).reshape(len(TARGET_MAG), len(TARGET_CI))
    #ax = FIG.add_subplot(GS[row, col])
    #ax.set_box_aspect(1)
    cont = ax.contourf(X, Y, z, levels=20, cmap='viridis')
    cb = FIG.colorbar(cont, ax=ax, fraction=0.046, pad=0.04, orientation="horizontal")
    cb.locator = mpl.ticker.MaxNLocator(nbins=5)
    cb.update_ticks()
    if row == len(PREFIX) - 1:
        ax.set_xlabel('A')
    if col == 0:
        ax.set_ylabel('CI')
    ax.set_title(column)
    return cont

for col in range(len(SUFFIX)):
    for row in range(len(PREFIX)):
        plot_contour(AXS[row, col], f'{PREFIX[row]}{SUFFIX[col]}')

plt.show()